# SV ablation: identifiability with and without SV scalar input

Tests whether removing the SV scalar from the observation hurts the model's ability to recover SV — i.e. whether SV is identifiable from pressure waveforms alone.

## Section 1 — Sim ablation (cv-sbi-sim)
- **v1** (`exp-v1_enc-cathlab_maf5_sims`): cath lab with SV, 809-dim, 1M sims
- **v1.1** (`exp-v1.1_enc-cathlab_maf5_sims`): cath lab without SV, 808-dim, 1M sims

Evaluated on 1000 held-out test sims (GT theta known). SV derived via S1 surrogate: θ → Vlv → SV.

## Section 2 — Real ablation (cv-dann-sbi)
- **v3** (`exp-v3_encoder-lipschitz_dann_flow-maf5`): joint training with SV, 809-dim
- **v3_nosv** (`exp-v3_nosv_encoder-lipschitz_dann_flow-maf5`): joint training without SV, 808-dim

Evaluated on 802 real patients. GT SV from `summaries/sv` (thermodilution/echo). SV derived via S1 surrogate.

In [ ]:
import json, sys, importlib.util, time
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.stats import pearsonr

try:
    ROOT_SIM = Path(globals()['_dh'][0]).parent
    assert (ROOT_SIM / 'dataset.py').exists()
except:
    ROOT_SIM = Path('/home/sa4604/cv-sbi-sim')

ROOT_DANN = Path('/home/sa4604/cv-dann-sbi')
sys.path.insert(0, str(ROOT_SIM))

from dataset import (
    load_stats, load_manifest,
    ReducedCVDataset, SurrogateDataset,
    PARAM_KEYS, PARAM_KEYS_INFER, WAVE_KEYS_CONT,
    N_CONT, T, _HR_IDX,
)
from models import ReducedAutoencoderEncoder, SurrogateDecoder

# Load LipschitzReducedAutoencoderEncoder from cv-dann-sbi via importlib.
# cv-dann-sbi/models.py imports from dataset — all shared constants are identical
# between repos, so resolving against cv-sbi-sim's dataset is safe.
_spec = importlib.util.spec_from_file_location('dann_models', ROOT_DANN / 'models.py')
_dann = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_dann)
LipschitzReducedAutoencoderEncoder = _dann.LipschitzReducedAutoencoderEncoder

SIM_ROOT  = Path('/media/local/SimData/hdf5/cv8/simset_10M_cv8Eed_20260314')
REAL_ROOT = Path('/home/sa4604/real_data/onebeat_300patients')

OUT_S1           = ROOT_SIM  / 'outputs/exp-v1_mlp-surrogate_sims'
OUT_V1           = ROOT_SIM  / 'outputs/exp-v1_enc-cathlab_maf5_sims'
OUT_V1_NOSV      = ROOT_SIM  / 'outputs/exp-v1.1_enc-cathlab_maf5_sims'
OUT_V3           = ROOT_DANN / 'outputs/exp-v3_encoder-lipschitz_dann_flow-maf5'
OUT_V3_NOSV      = ROOT_DANN / 'outputs/exp-v3_nosv_encoder-lipschitz_dann_flow-maf5'

CACHE_DIR = ROOT_SIM / 'outputs/ablation_nosv_cache'
CACHE_DIR.mkdir(exist_ok=True)

STATS_PATH      = ROOT_SIM  / 'norm_stats.json'
STATS_DANN_PATH = ROOT_DANN / 'norm_stats.json'

N_TEST    = 1000
N_SAMPLES = 500
device    = torch.device('cuda:0')

stats      = load_stats(STATS_PATH)
stats_dann = json.load(open(STATS_DANN_PATH))
manifest   = load_manifest(SIM_ROOT / 'manifest_test.json')
manifest_train = load_manifest(SIM_ROOT / 'manifest_train.json')

prior_lo  = np.array([manifest_train['config']['pvar_low'][k]  for k in PARAM_KEYS_INFER])
prior_hi  = np.array([manifest_train['config']['pvar_high'][k] for k in PARAM_KEYS_INFER])
prior_std = (prior_hi - prior_lo) / np.sqrt(12)

vlv_idx = WAVE_KEYS_CONT.index('Vlv')
vlv_std = stats['waves']['Vlv']['std']

print(f'ROOT_SIM:  {ROOT_SIM}')
print(f'ROOT_DANN: {ROOT_DANN}')
print(f'device:    {device}')

In [ ]:
# S1 surrogate + shared helpers
with open(OUT_S1 / 'theta_norm.json') as f:
    theta_norm = json.load(f)
theta_norm_mean = torch.tensor(theta_norm['mean'], dtype=torch.float32).to(device)
theta_norm_std  = torch.tensor(theta_norm['std'],  dtype=torch.float32).to(device)

surrogate = SurrogateDecoder(hidden=512, n_layers=4).to(device)
surrogate.load_state_dict(torch.load(OUT_S1 / 'decoder.pt', map_location=device))
surrogate.eval()
print('Surrogate:', surrogate.describe())


def run_inference_sim(label, encoder, flow_net, dataset, cache_path):
    """Run flow on N_TEST sims → theta_true (N,24) and samples (N,N_SAMPLES,24)."""
    if cache_path.exists():
        d = np.load(cache_path)
        print(f'[{label}] loaded from cache')
        return d['theta_true'], d['samples']
    print(f'[{label}] running inference on {N_TEST} sims...')
    theta_list, samples_list = [], []
    t0 = time.time()
    for i in range(N_TEST):
        theta, x = dataset[i]
        with torch.no_grad():
            z = encoder(x.unsqueeze(0).to(device))
            s = flow_net.sample((N_SAMPLES,), condition=z).squeeze(1).cpu()
        theta_list.append(theta.numpy())
        samples_list.append(s.numpy())
        if i % 200 == 0:
            print(f'  {i}/{N_TEST}  ({time.time()-t0:.0f}s)', end='\r', flush=True)
    theta_true = np.stack(theta_list)
    samples    = np.stack(samples_list)
    np.savez(cache_path, theta_true=theta_true, samples=samples)
    print(f'[{label}] done  ({time.time()-t0:.0f}s)')
    return theta_true, samples


def run_inference_real(label, encoder, flow_net, beats, cache_path):
    """Run flow on real beats tensor (N,obs_dim) → samples (N,N_SAMPLES,24)."""
    if cache_path.exists():
        d = np.load(cache_path)
        print(f'[{label}] loaded from cache')
        return d['samples']
    print(f'[{label}] running inference on {len(beats)} real beats...')
    samples_list = []
    t0 = time.time()
    for i, x in enumerate(beats):
        with torch.no_grad():
            z = encoder(x.unsqueeze(0))
            s = flow_net.sample((N_SAMPLES,), condition=z).squeeze(1).cpu()
        samples_list.append(s.numpy())
        if i % 100 == 0:
            print(f'  {i}/{len(beats)}  ({time.time()-t0:.0f}s)', end='\r', flush=True)
    samples = np.stack(samples_list)
    np.savez(cache_path, samples=samples)
    print(f'[{label}] done  ({time.time()-t0:.0f}s)')
    return samples


def derive_sv(samples_24, hr_arr):
    """samples_24: (N, N_SAMPLES, 24); hr_arr: (N,) raw HR → SV (N, N_SAMPLES) in mL."""
    N = len(samples_24)
    sv_out = np.zeros((N, N_SAMPLES))
    for i in range(N):
        s24 = torch.from_numpy(samples_24[i]).float()
        hr_col = torch.full((N_SAMPLES, 1), float(hr_arr[i]))
        s25 = torch.cat([s24[:, :_HR_IDX], hr_col, s24[:, _HR_IDX:]], dim=1)
        s25_z = (s25.to(device) - theta_norm_mean) / theta_norm_std
        with torch.no_grad():
            waves_z = surrogate(s25_z)
        vlv_z = waves_z.cpu().numpy().reshape(N_SAMPLES, N_CONT, T)[:, vlv_idx, :]
        sv_out[i] = (vlv_z.max(axis=1) - vlv_z.min(axis=1)) * vlv_std
    return sv_out

---
## Section 1 — Sim ablation
v1 (with SV, 809-dim) vs v1.1 (no SV, 808-dim) on 1000 held-out test sims.

In [ ]:
# Datasets
ds_v1     = ReducedCVDataset(str(SIM_ROOT / 'test'), manifest['index'][:N_TEST], stats, include_sv=True)
ds_v1_nosv = ReducedCVDataset(str(SIM_ROOT / 'test'), manifest['index'][:N_TEST], stats, include_sv=False)

# v1 encoder + flow (with SV)
enc_v1 = ReducedAutoencoderEncoder(latent_dim=128, n_scalars=5).to(device)
enc_v1.load_state_dict(torch.load(OUT_V1 / 'encoder.pt', map_location=device))
enc_v1.eval()
flow_v1 = torch.load(OUT_V1 / 'flow_net.pt', map_location=device, weights_only=False)
flow_v1.eval()

# v1.1 encoder + flow (no SV)
enc_v1_nosv = ReducedAutoencoderEncoder(latent_dim=128, n_scalars=4).to(device)
enc_v1_nosv.load_state_dict(torch.load(OUT_V1_NOSV / 'encoder.pt', map_location=device))
enc_v1_nosv.eval()
flow_v1_nosv = torch.load(OUT_V1_NOSV / 'flow_net.pt', map_location=device, weights_only=False)
flow_v1_nosv.eval()

theta_v1,     samp_v1     = run_inference_sim('v1',     enc_v1,     flow_v1,     ds_v1,      CACHE_DIR / 'sim_v1.npz')
theta_v1_nosv, samp_v1_nosv = run_inference_sim('v1.1-nosv', enc_v1_nosv, flow_v1_nosv, ds_v1_nosv, CACHE_DIR / 'sim_v1_nosv.npz')

del enc_v1, flow_v1, enc_v1_nosv, flow_v1_nosv; torch.cuda.empty_cache()
ds_v1.close(); ds_v1_nosv.close()

# HR and GT SV from test sims
ds_surr = SurrogateDataset(str(SIM_ROOT / 'test'), manifest['index'][:N_TEST], stats)
hr_sim, sv_sim_gt = np.zeros(N_TEST), np.zeros(N_TEST)
for i in range(N_TEST):
    theta_raw, waves_z = ds_surr[i]
    hr_sim[i] = theta_raw[_HR_IDX].item()
    vlv_z = waves_z.numpy().reshape(N_CONT, T)[vlv_idx]
    sv_sim_gt[i] = (vlv_z.max() - vlv_z.min()) * vlv_std
ds_surr.close()

print(f'theta shapes: {theta_v1.shape}, {theta_v1_nosv.shape}')
print(f'GT SV range: {sv_sim_gt.min():.1f}–{sv_sim_gt.max():.1f} mL')

In [ ]:
# Identifiability ratios: posterior std / prior std
post_std_v1     = samp_v1.std(axis=1).mean(axis=0)      # (24,)
post_std_v1_nosv = samp_v1_nosv.std(axis=1).mean(axis=0)

ident_v1     = post_std_v1     / prior_std
ident_v1_nosv = post_std_v1_nosv / prior_std

order = np.argsort(ident_v1)  # sorted by v1 identifiability
params_sorted = [PARAM_KEYS_INFER[i] for i in order]

x = np.arange(len(PARAM_KEYS_INFER))
w = 0.35

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x - w/2, ident_v1[order],     w, label='v1 (with SV)',  color='tomato',    alpha=0.85)
ax.bar(x + w/2, ident_v1_nosv[order], w, label='v1.1 (no SV)', color='steelblue', alpha=0.85)
ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--', label='prior (uninformative)')
ax.set_xticks(x); ax.set_xticklabels(params_sorted, rotation=55, ha='right', fontsize=8)
ax.set_ylabel('Posterior std / prior std  (lower = more identified)')
ax.set_title('Identifiability ratio: with SV vs without SV — sim flow (1M sims)', fontsize=12)
ax.legend(fontsize=10); ax.set_ylim(0, 1.15)
plt.tight_layout(); plt.show()

# R² per parameter
means_v1     = samp_v1.mean(axis=1)
means_v1_nosv = samp_v1_nosv.mean(axis=1)
r2_v1, r2_v1_nosv = np.zeros(24), np.zeros(24)
for i in range(24):
    t = theta_v1[:, i]
    r2_v1[i]     = pearsonr(t, means_v1[:, i])[0]**2
    r2_v1_nosv[i] = pearsonr(t, means_v1_nosv[:, i])[0]**2

print(f'\n{"Parameter":<14} {"v1 ratio":>10} {"v1.1 ratio":>12} {"v1 R²":>8} {"v1.1 R²":>9}')
print('-' * 58)
for i in order:
    name = PARAM_KEYS_INFER[i]
    delta = ident_v1_nosv[i] - ident_v1[i]
    flag  = '  ↑' if delta > 0.05 else ''
    print(f'{name:<14} {ident_v1[i]:>10.3f} {ident_v1_nosv[i]:>12.3f} '
          f'{r2_v1[i]:>8.3f} {r2_v1_nosv[i]:>9.3f}{flag}')
print(f'\nMean ratio:  v1={ident_v1.mean():.3f}  v1.1={ident_v1_nosv.mean():.3f}')

In [ ]:
# Derive SV from posteriors via S1
SV_SIM_CACHE = CACHE_DIR / 'sv_sim_ablation.npz'
if SV_SIM_CACHE.exists():
    d = np.load(SV_SIM_CACHE)
    sv_v1, sv_v1_nosv = d['sv_v1'], d['sv_v1_nosv']
    print('SV posteriors loaded from cache.')
else:
    print('Deriving SV from v1 posterior...')
    sv_v1 = derive_sv(samp_v1, hr_sim)
    print('Deriving SV from v1.1 posterior...')
    sv_v1_nosv = derive_sv(samp_v1_nosv, hr_sim)
    np.savez(SV_SIM_CACHE, sv_v1=sv_v1, sv_v1_nosv=sv_v1_nosv)
    print('Cached.')

sv_mean_v1     = sv_v1.mean(axis=1)
sv_mean_v1_nosv = sv_v1_nosv.mean(axis=1)
sv_std_v1      = sv_v1.std(axis=1)
sv_std_v1_nosv  = sv_v1_nosv.std(axis=1)

r2_sv_v1     = pearsonr(sv_sim_gt, sv_mean_v1)[0]**2
r2_sv_v1_nosv = pearsonr(sv_sim_gt, sv_mean_v1_nosv)[0]**2
mape_sv_v1     = np.mean(np.abs(sv_mean_v1     - sv_sim_gt) / (np.abs(sv_sim_gt) + 1e-9)) * 100
mape_sv_v1_nosv = np.mean(np.abs(sv_mean_v1_nosv - sv_sim_gt) / (np.abs(sv_sim_gt) + 1e-9)) * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, label, sv_mean, color, r2, mape in [
    (axes[0], 'v1 (with SV)',  sv_mean_v1,     'tomato',    r2_sv_v1,     mape_sv_v1),
    (axes[1], 'v1.1 (no SV)', sv_mean_v1_nosv, 'steelblue', r2_sv_v1_nosv, mape_sv_v1_nosv),
]:
    lo, hi = min(sv_sim_gt.min(), sv_mean.min()), max(sv_sim_gt.max(), sv_mean.max())
    ax.scatter(sv_sim_gt, sv_mean, s=8, alpha=0.5, color=color)
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.8)
    ax.set_xlabel('GT SV (mL)'); ax.set_ylabel('Posterior mean SV (mL)')
    ax.set_title(f'{label}\nR²={r2:.3f}  MAPE={mape:.1f}%', fontsize=10)

ax = axes[2]
bins = np.linspace(0, max(sv_std_v1.max(), sv_std_v1_nosv.max()) * 1.05, 40)
ax.hist(sv_std_v1,      bins=bins, color='tomato',    alpha=0.6, label=f'v1 (with SV)  mean={sv_std_v1.mean():.1f} mL')
ax.hist(sv_std_v1_nosv, bins=bins, color='steelblue', alpha=0.6, label=f'v1.1 (no SV)  mean={sv_std_v1_nosv.mean():.1f} mL')
ax.set_xlabel('SV posterior std (mL)'); ax.set_ylabel('Count')
ax.set_title('SV posterior uncertainty — sim', fontsize=10); ax.legend(fontsize=9)

fig.suptitle('SV identifiability: sim ablation (θ → S1 → Vlv → SV)', fontsize=12)
plt.tight_layout(); plt.show()

print(f'SV summary (sims):')
print(f'  v1   (with SV):  R²={r2_sv_v1:.3f}  MAPE={mape_sv_v1:.1f}%  post_std={sv_std_v1.mean():.1f} mL')
print(f'  v1.1 (no SV):   R²={r2_sv_v1_nosv:.3f}  MAPE={mape_sv_v1_nosv:.1f}%  post_std={sv_std_v1_nosv.mean():.1f} mL')

---
## Section 2 — Real ablation (cv-dann-sbi)
v3 (with SV) vs v3_nosv (no SV) on 802 real patients. Both use legacy scalar norm (sim stats).
GT SV from `summaries/sv` in the real H5 files (thermodilution/echo measurement).

In [ ]:
# Normalization constants for real beats — legacy mode (same as v3 and v3_nosv training)
WAVE_KEYS_REAL = ['Prv', 'Pra', 'Pvp', 'Pap']
w_dann = stats_dann['waves']
p_dann = stats_dann['parameters']

wave_mean_dann = torch.tensor([w_dann[k]['mean'] for k in WAVE_KEYS_REAL], dtype=torch.float32).unsqueeze(1)
wave_std_dann  = torch.tensor([w_dann[k]['std']  for k in WAVE_KEYS_REAL], dtype=torch.float32).unsqueeze(1)
pas_mean  = w_dann['Pas']['mean'];  pas_std  = w_dann['Pas']['std']  + 1e-8
sv_legacy = w_dann['Vlv']['std']   + 1e-8   # legacy: sv_ml / vlv_std, mean=0
hr_mean   = p_dann['HR']['mean'];   hr_std   = p_dann['HR']['std']   + 1e-8

beats_with_sv, beats_no_sv = [], []
hr_real, sv_gt = [], []

for fpath in sorted(REAL_ROOT.glob('*.h5')):
    with h5py.File(fpath, 'r') as f:
        for key in sorted(f.keys()):
            if not key.startswith('beat_'):
                continue
            g = f[key]
            waves = np.stack([g[f'waves/{k}'][:].astype(np.float32) for k in WAVE_KEYS_REAL])
            wt    = (torch.from_numpy(waves) - wave_mean_dann) / (wave_std_dann + 1e-8)
            wflat = wt.reshape(-1)  # (804,)

            map_ = float(g['summaries/map'][()])
            sbp  = float(g['summaries/sbp'][()])
            dbp  = float(g['summaries/dbp'][()])
            sv   = float(g['summaries/sv'][()])
            hr   = float(g['parameters/HR'][()])

            map_z = (map_ - pas_mean) / pas_std
            sbp_z = (sbp  - pas_mean) / pas_std
            dbp_z = (dbp  - pas_mean) / pas_std
            sv_z  = sv / sv_legacy
            hr_z  = (hr  - hr_mean)  / hr_std

            beats_with_sv.append(torch.cat([wflat, torch.tensor([map_z, sbp_z, dbp_z, sv_z, hr_z])]))
            beats_no_sv.append(torch.cat([wflat, torch.tensor([map_z, sbp_z, dbp_z, hr_z])]))
            hr_real.append(hr)
            sv_gt.append(sv)

beats_with_sv = torch.stack(beats_with_sv).to(device)  # (N_real, 809)
beats_no_sv   = torch.stack(beats_no_sv).to(device)    # (N_real, 808)
hr_real = np.array(hr_real)
sv_gt   = np.array(sv_gt)
N_real  = len(sv_gt)

print(f'Loaded {N_real} real beats')
print(f'GT SV:  {sv_gt.min():.1f}–{sv_gt.max():.1f} mL  mean={sv_gt.mean():.1f}±{sv_gt.std():.1f}')
print(f'beats_with_sv: {tuple(beats_with_sv.shape)}  beats_no_sv: {tuple(beats_no_sv.shape)}')

In [ ]:
# Load v3 and v3_nosv encoders + flows
enc_v3 = LipschitzReducedAutoencoderEncoder(latent_dim=128, n_scalars=5).to(device)
enc_v3.load_state_dict(torch.load(OUT_V3 / 'encoder.pt', map_location=device))
enc_v3.eval()
flow_v3 = torch.load(OUT_V3 / 'flow_net.pt', map_location=device, weights_only=False)
flow_v3.eval()

enc_v3_nosv = LipschitzReducedAutoencoderEncoder(latent_dim=128, n_scalars=4).to(device)
enc_v3_nosv.load_state_dict(torch.load(OUT_V3_NOSV / 'encoder.pt', map_location=device))
enc_v3_nosv.eval()
flow_v3_nosv = torch.load(OUT_V3_NOSV / 'flow_net.pt', map_location=device, weights_only=False)
flow_v3_nosv.eval()

samp_v3     = run_inference_real('v3',      enc_v3,     flow_v3,     beats_with_sv, CACHE_DIR / 'real_v3.npz')
samp_v3_nosv = run_inference_real('v3_nosv', enc_v3_nosv, flow_v3_nosv, beats_no_sv,   CACHE_DIR / 'real_v3_nosv.npz')

del enc_v3, flow_v3, enc_v3_nosv, flow_v3_nosv; torch.cuda.empty_cache()
print(f'samples shape: {samp_v3.shape}')

In [ ]:
# Derive SV from real posteriors via S1
SV_REAL_CACHE = CACHE_DIR / 'sv_real_ablation.npz'
if SV_REAL_CACHE.exists():
    d = np.load(SV_REAL_CACHE)
    sv_v3, sv_v3_nosv = d['sv_v3'], d['sv_v3_nosv']
    print('SV posteriors loaded from cache.')
else:
    print('Deriving SV from v3 posterior...')
    sv_v3 = derive_sv(samp_v3, hr_real)
    print('Deriving SV from v3_nosv posterior...')
    sv_v3_nosv = derive_sv(samp_v3_nosv, hr_real)
    np.savez(SV_REAL_CACHE, sv_v3=sv_v3, sv_v3_nosv=sv_v3_nosv)
    print('Cached.')

sv_mean_v3     = sv_v3.mean(axis=1)
sv_mean_v3_nosv = sv_v3_nosv.mean(axis=1)
sv_std_v3      = sv_v3.std(axis=1)
sv_std_v3_nosv  = sv_v3_nosv.std(axis=1)

r2_sv_v3     = pearsonr(sv_gt, sv_mean_v3)[0]**2
r2_sv_v3_nosv = pearsonr(sv_gt, sv_mean_v3_nosv)[0]**2
mape_sv_v3     = np.mean(np.abs(sv_mean_v3     - sv_gt) / (np.abs(sv_gt) + 1e-9)) * 100
mape_sv_v3_nosv = np.mean(np.abs(sv_mean_v3_nosv - sv_gt) / (np.abs(sv_gt) + 1e-9)) * 100

# Posterior std / prior std for all params (real patients, no GT theta)
post_std_v3     = samp_v3.std(axis=1).mean(axis=0)      # (24,)
post_std_v3_nosv = samp_v3_nosv.std(axis=1).mean(axis=0)
ident_v3     = post_std_v3     / prior_std
ident_v3_nosv = post_std_v3_nosv / prior_std

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, label, sv_mean, color, r2, mape in [
    (axes[0], 'v3 (with SV)',  sv_mean_v3,     'tomato',    r2_sv_v3,     mape_sv_v3),
    (axes[1], 'v3_nosv (no SV)', sv_mean_v3_nosv, 'steelblue', r2_sv_v3_nosv, mape_sv_v3_nosv),
]:
    lo, hi = min(sv_gt.min(), sv_mean.min()), max(sv_gt.max(), sv_mean.max())
    ax.scatter(sv_gt, sv_mean, s=12, alpha=0.6, color=color)
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.8)
    ax.set_xlabel('GT SV — thermodilution/echo (mL)'); ax.set_ylabel('Posterior mean SV (mL)')
    ax.set_title(f'{label}\nR²={r2:.3f}  MAPE={mape:.1f}%', fontsize=10)

ax = axes[2]
bins = np.linspace(0, max(sv_std_v3.max(), sv_std_v3_nosv.max()) * 1.05, 40)
ax.hist(sv_std_v3,      bins=bins, color='tomato',    alpha=0.6, label=f'v3 (with SV)   mean={sv_std_v3.mean():.1f} mL')
ax.hist(sv_std_v3_nosv, bins=bins, color='steelblue', alpha=0.6, label=f'v3_nosv (no SV) mean={sv_std_v3_nosv.mean():.1f} mL')
ax.set_xlabel('SV posterior std (mL)'); ax.set_ylabel('Count')
ax.set_title('SV posterior uncertainty — real patients', fontsize=10); ax.legend(fontsize=9)

fig.suptitle('SV identifiability: real patient ablation (θ → S1 → Vlv → SV)', fontsize=12)
plt.tight_layout(); plt.show()

print(f'SV summary (real patients):')
print(f'  v3       (with SV):  R²={r2_sv_v3:.3f}  MAPE={mape_sv_v3:.1f}%  post_std={sv_std_v3.mean():.1f} mL')
print(f'  v3_nosv  (no SV):   R²={r2_sv_v3_nosv:.3f}  MAPE={mape_sv_v3_nosv:.1f}%  post_std={sv_std_v3_nosv.mean():.1f} mL')

In [ ]:
# Identifiability ratio on real patients (posterior std / prior std, no GT theta)
order_real = np.argsort(ident_v3)
params_sorted_real = [PARAM_KEYS_INFER[i] for i in order_real]

x = np.arange(len(PARAM_KEYS_INFER))
w = 0.35

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x - w/2, ident_v3[order_real],     w, label='v3 (with SV)',  color='tomato',    alpha=0.85)
ax.bar(x + w/2, ident_v3_nosv[order_real], w, label='v3_nosv (no SV)', color='steelblue', alpha=0.85)
ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--', label='prior (uninformative)')
ax.set_xticks(x); ax.set_xticklabels(params_sorted_real, rotation=55, ha='right', fontsize=8)
ax.set_ylabel('Posterior std / prior std  (lower = more identified)')
ax.set_title('Identifiability ratio on real patients: v3 vs v3_nosv', fontsize=12)
ax.legend(fontsize=10); ax.set_ylim(0, 1.15)
plt.tight_layout(); plt.show()

print(f'\n{"Parameter":<14} {"v3 ratio":>10} {"v3_nosv ratio":>14}')
print('-' * 42)
for i in order_real:
    name = PARAM_KEYS_INFER[i]
    delta = ident_v3_nosv[i] - ident_v3[i]
    flag  = '  ↑' if delta > 0.05 else ''
    print(f'{name:<14} {ident_v3[i]:>10.3f} {ident_v3_nosv[i]:>14.3f}{flag}')
print(f'\nMean ratio:  v3={ident_v3.mean():.3f}  v3_nosv={ident_v3_nosv.mean():.3f}')

In [ ]:
# Summary table: sim and real side by side
print('=' * 70)
print('SUMMARY: SV ABLATION')
print('=' * 70)
print(f'{"":30s} {"with SV":>12} {"no SV":>12} {"delta":>8}')
print('-' * 66)

rows = [
    ('Sim: val NLL',          '-18.54',        '-13.53',        ''),
    ('Sim: SV R²',             f'{r2_sv_v1:.3f}',  f'{r2_sv_v1_nosv:.3f}',  f'{r2_sv_v1_nosv - r2_sv_v1:+.3f}'),
    ('Sim: SV MAPE (%)',        f'{mape_sv_v1:.1f}',  f'{mape_sv_v1_nosv:.1f}',  ''),
    ('Sim: SV post_std (mL)',   f'{sv_std_v1.mean():.1f}',   f'{sv_std_v1_nosv.mean():.1f}',   ''),
    ('Sim: mean ident ratio',  f'{ident_v1.mean():.3f}',  f'{ident_v1_nosv.mean():.3f}',  f'{ident_v1_nosv.mean() - ident_v1.mean():+.3f}'),
    ('', '', '', ''),
    ('Real: SV R²',            f'{r2_sv_v3:.3f}',  f'{r2_sv_v3_nosv:.3f}',  f'{r2_sv_v3_nosv - r2_sv_v3:+.3f}'),
    ('Real: SV MAPE (%)',       f'{mape_sv_v3:.1f}',  f'{mape_sv_v3_nosv:.1f}',  ''),
    ('Real: SV post_std (mL)', f'{sv_std_v3.mean():.1f}',  f'{sv_std_v3_nosv.mean():.1f}',  ''),
    ('Real: mean ident ratio', f'{ident_v3.mean():.3f}',  f'{ident_v3_nosv.mean():.3f}',  f'{ident_v3_nosv.mean() - ident_v3.mean():+.3f}'),
]
for label, val_with, val_no, delta in rows:
    print(f'{label:<30s} {val_with:>12} {val_no:>12} {delta:>8}')
print('=' * 70)
print('Interpretation: if SV R² drops sharply without SV input, SV is not')
print('recoverable from pressure waveforms alone. If R² holds, SV is redundant.')